# SortStudio research

Start with 1,000 seeded uniform integer records. The CPU pass verifies the 20 SortStudio implementations
and tests hybrid candidates. The optional A100 pass compares two GPU implementations that return stable
indices into the same CSV. Record IDs distinguish equal keys and allow the checks to detect lost or
duplicated records. The runs are indexed by commit, configuration, and input checksum.

The CPU benchmark caps workers at 12 and measures sort latency separately from parallel verification.
GPU tables separate device-stream time from wall time that includes transfers. A small uniform run
supports a claim about that configuration. Claims about other workloads require held-out distributions,
multiple sizes, and reliable measurements of any candidate that appeared to win.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

REPO = "https://github.com/NicoleBGil/SortStudio.git"
REVISION = "994f44f6b1d9ba95677360bd8976b93d97a24bb0"
RECORDS = 1000
SEED = 138
logical_cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
WORKERS = min(12, logical_cpus)
if shutil.which("node") is None:
    raise RuntimeError("Select a runtime with Node.js available to run the original 20-method engine.")
work = Path(tempfile.mkdtemp(prefix="sortstudio-"))
subprocess.run(["git", "init", "-q", str(work)], check=True)
subprocess.run(["git", "-C", str(work), "fetch", "--quiet", "--depth=1", REPO, REVISION], check=True)
subprocess.run(["git", "-C", str(work), "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)
seen_revision = subprocess.check_output(["git", "-C", str(work), "rev-parse", "HEAD"], text=True).strip()
if seen_revision != REVISION:
    raise RuntimeError("Unexpected SortStudio revision")
run = work / "trial-output"
print("SortStudio", seen_revision, "| workers", WORKERS)

In [ ]:
import csv
import numpy as np

completed = subprocess.run(
    ["node", "research/sort-trials.cjs", "--workers", str(WORKERS),
     "--records", str(RECORDS), "--seed", str(SEED), "--output", str(run)],
    cwd=work, check=True, text=True, stdout=subprocess.PIPE,
)
print(completed.stdout)
records_file = run / "tuning-input.csv"
with records_file.open(newline="", encoding="utf-8") as source:
    records = list(csv.DictReader(source))
ids = [int(row["id"]) for row in records]
assert ids == list(range(RECORDS))
keys = np.array([int(row["key"]) for row in records], dtype=np.int32)
reference = np.argsort(keys, kind="stable")
source_hash = hashlib.sha256(records_file.read_bytes()).hexdigest()
print("Same input for CPU and GPU.", RECORDS, "records. SHA-256", source_hash)
print("CPU evidence", run / "trials.json")

## Run the A100 comparison

Choose an A100 in Colab’s runtime settings and run the next cell. It checks the hardware before
collecting GPU results. GPU availability depends on the managed runtime and account.
[Colab resource limits](https://research.google.com/colaboratory/faq.html#resource-limits)

The unsigned key candidate flips the sign bit to preserve signed order, packs the record's original
position after the key, then sorts the combined word and decodes the positions. Sorting equal keys
by position preserves stability. Check both GPU outputs against the independent stable NumPy
reference. GPU events are synchronized before each device timing is read.
[CuPy timing](https://docs.cupy.dev/en/stable/user_guide/performance.html)

In [ ]:
import platform
import time

try:
    hardware = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        text=True,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError("Attach a Colab A100 runtime to run the GPU cells. CPU output is already retained.") from error

if "A100" not in hardware:
    raise RuntimeError("This GPU comparison requires an A100. Detected: " + hardware)

try:
    import cupy as cp
except ModuleNotFoundError as error:
    raise RuntimeError("Install the CuPy wheel matching this runtime’s CUDA major, then rerun the cell.") from error

device = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
device_name = device["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode("utf-8")
if "A100" not in device_name:
    raise RuntimeError("The current CUDA device is not an A100: " + device_name)
position_bits = max(1, (len(keys) - 1).bit_length())
if 32 + position_bits > 64:
    raise ValueError("The signed key and position together exceed 64 bits")
position_mask = cp.uint64((1 << position_bits) - 1)

def stable_argsort(device_keys):
    return cp.argsort(device_keys)

def packed_key_sort(device_keys):
    ordered_keys = (device_keys.astype(cp.uint32) ^ cp.uint32(0x80000000)).astype(cp.uint64)
    positions = cp.arange(device_keys.size, dtype=cp.uint64)
    combined = (ordered_keys << cp.uint64(position_bits)) | positions
    return (cp.sort(combined) & position_mask).astype(cp.int64)

device_keys = cp.asarray(keys)
strategies = {"cupy_stable_argsort": stable_argsort, "cupy_packed_signed_key": packed_key_sort}
for name, sort in strategies.items():
    actual = cp.asnumpy(sort(device_keys))
    np.testing.assert_array_equal(actual, reference, err_msg=name)

samples = {name: {"cuda_stream_ms": [], "host_with_transfers_ms": []} for name in strategies}
for _ in range(8):
    for sort in strategies.values():
        cp.asnumpy(sort(cp.asarray(keys)))
for repeat in range(40):
    cycle = list(strategies.items())
    if repeat % 2:
        cycle.reverse()
    for name, sort in cycle:
        start, end = cp.cuda.Event(), cp.cuda.Event()
        start.record()
        result = sort(device_keys)
        end.record()
        end.synchronize()
        samples[name]["cuda_stream_ms"].append(cp.cuda.get_elapsed_time(start, end))
        started = time.perf_counter()
        copied = cp.asnumpy(sort(cp.asarray(keys)))
        samples[name]["host_with_transfers_ms"].append((time.perf_counter() - started) * 1000)
        np.testing.assert_array_equal(copied, reference)

report = {
    "revision": REVISION, "dataset_sha256": source_hash, "records": len(keys), "seed": SEED,
    "gpu": device_name, "nvidia_smi": hardware, "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "cupy": cp.__version__, "python": platform.python_version(),
    "scope": "Warm, serial sorts of stable original indices using the same 1,000-record fixture.",
    "samples": samples,
    "median_ms": {name: {metric: float(np.median(values)) for metric, values in data.items()}
                  for name, data in samples.items()},
}
(run / "gpu-a100.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report["median_ms"], indent=2))

In [ ]:
# Download a single archive containing the recorded input and all observed results.
destination = shutil.make_archive(str(work / "sortstudio-results"), "zip", root_dir=run)
print(destination)
from google.colab import files
files.download(destination)